In [ ]:
!pip install -q rapidfuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 16.5 MB/s eta 0:00:00


In [ ]:
import pandas as pd
import re
from rapidfuzz import process, fuzz
from tqdm import tqdm

tqdm.pandas()

# 학과명 정제 함수
def normalize_major(text):
    if pd.isna(text):
        return ''
    text = str(text)
    text = re.sub(r'\(.*?\)', '', text)
    text = re.sub(r'[^가-힣A-Za-z0-9]', '', text)
    return text.strip()

# fuzzy 유사도 매칭 함수
def get_closest_major(major_name, mapping_majors_list, major_to_dept):
    match, score, _ = process.extractOne(major_name, mapping_majors_list, scorer=fuzz.token_sort_ratio)
    if score >= 80:
        dept = major_to_dept.get(match, None)
        return pd.Series([dept, match, score])
    else:
        return pd.Series([None, None, score])

df_all_years = []

for year in range(2014, 2021):
    print(f"{year}년")

    # 파일명 정의
    df_file = f"학과별_장학금_{year}.csv"
    mapping_file = f"{year}_기준.csv"

    # 데이터 로딩 및 정제
    df = pd.read_csv(df_file)
    df = df.dropna(axis=1, how='all')
    df.columns = ['기준연도','학교종류', '학교명', '학과명', '구분', '재학생', "1인당장학금"]

    # 필터링
    c1 = df['학교종류'].str.contains('사이버대학', na=False)
    c2 = df['구분'].str.contains('원격', na=False)
    c3 = df['재학생'] == 0
    df = df[~(c1 | c2 | c3)]

    # 열 정리
    df = df.drop(['학교종류', '구분'], axis=1)
    df.columns = ['기준연도', '학교명', '학과명', '재학생', '1인당장학금']

    # 결측값 처리
    df['기준연도'] = df['기준연도'].fillna(method='ffill')
    df['학교명'] = df['학교명'].fillna(method='ffill')
    df = df.dropna(subset=['학과명', '1인당장학금'])

    # 형 변환
    df['기준연도'] = df['기준연도'].astype(int)
    df['재학생'] = pd.to_numeric(df['재학생'].astype(str).str.replace(',', ''), errors='coerce')
    df['1인당장학금'] = pd.to_numeric(df['1인당장학금'].astype(str).str.replace(',', ''), errors='coerce')
    df = df.reset_index(drop=True)

    # 학과명 정제
    df['학과명'] = df['학과명'].apply(normalize_major)
    df['학교명'] = df['학교명'].astype(str).str.strip()

    # 기준 매핑 로딩 및 정제
    df_mapping = pd.read_csv(mapping_file)
    df_mapping['학교명'] = df_mapping['학교명'].astype(str).str.strip()
    df_mapping['학과명'] = df_mapping['학과명'].apply(normalize_major)

    # 단일 학과 매핑
    major_group = df_mapping.groupby('학과명')['대계열'].nunique()
    single_majors = major_group[major_group == 1].index.tolist()
    df_mapping_single = df_mapping[df_mapping['학과명'].isin(single_majors)].drop_duplicates(subset=['학과명'])[['학과명', '대계열']]
    df_merged = pd.merge(df, df_mapping_single, how='left', on='학과명')

    # 복수 계열: 학교명+학과명으로 추가 매핑
    df_remaining = df_merged[df_merged['대계열'].isna()].drop(columns=['대계열'])
    df_mapping_multi = df_mapping[~df_mapping['학과명'].isin(single_majors)].drop_duplicates(subset=['학교명', '학과명'])
    df_remaining_merged = pd.merge(df_remaining, df_mapping_multi, how='left', on=['학교명', '학과명'])

    # 통합
    df_combined = pd.concat([
        df_merged[df_merged['대계열'].notna()],
        df_remaining_merged
    ], ignore_index=True)

    # fuzzy 매칭 대상
    df_unmatched = df_combined[df_combined['대계열'].isna()].copy()

    # 매핑 리스트
    mapping_majors_list = df_mapping['학과명'].dropna().unique().tolist()
    major_to_dept = df_mapping.drop_duplicates('학과명').set_index('학과명')['대계열'].to_dict()

    # 유사도 기반 매칭 (apply로 진행)
    df_unmatched[['대계열', '추천_학과명', '유사도']] = df_unmatched['학과명'].apply(
        lambda x: get_closest_major(x, mapping_majors_list, major_to_dept)
    )

    # 최종 연도별 통합
    df_final = pd.concat([
        df_combined[df_combined['대계열'].notna()],
        df_unmatched[df_unmatched['대계열'].notna()],
        df_unmatched[df_unmatched['대계열'].isna()]
    ], ignore_index=True)

    df_all_years.append(df_final)

# 전체 연도 통합
df_total = pd.concat(df_all_years, ignore_index=True)
df_total = df_total.dropna(axis=1, how='all')

# 저장
df_total.to_csv("학과별_장학금_14_20_재분류.csv", index=False, encoding='utf-8-sig')

KeyboardInterrupt: 